In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
def remove_unwanted_columns(df):
    """
    Removes specific unwanted columns if they exist in the DataFrame.
    """
    unwanted = [
        'Unnamed: 0', 'Month', 'day_of_week', 
        'Day_of_week', 'pm25_lag1', 'pm25_lag7', 'to_date',
    ]
    
    # Only drop columns that are actually present in the dataframe
    cols_to_drop = [col for col in unwanted if col in df.columns]
    
    return df.drop(columns=cols_to_drop)



def fix_temporal_structure(df, datetime_col='from_date', freq='h'):
    """
    Ensures the dataframe has a continuous datetime index with no gaps.
    """
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    
    # Remove duplicates by taking the mean of entries with the same timestamp
    df = df.groupby(datetime_col).mean(numeric_only=True).reset_index()

    # Set index and fill missing hours
    df = df.set_index(datetime_col)
    df = df.asfreq(freq)
    
    return df



def apply_physical_bounds(df):
    """
    Clips sensor data to realistic physical limits.
    """
    # Pollutants & Wind Speed: Cannot be negative
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].clip(lower=0)
    
    # Humidity: Cannot exceed 100%
    if 'humidity' in df.columns:
        df['humidity'] = df['humidity'].clip(upper=100)
        
    return df



def handle_missing_values(df, max_gap=3):
    """
    Fills gaps using linear interpolation. 
    'max_gap' limits interpolation so we don't 'guess' too much data.
    """
    # Interpolate small gaps linearly
    df = df.interpolate(method='linear', limit=max_gap)
    
    # For larger gaps, use a backfill/forward fill to catch edges
    df = df.ffill().bfill()
    
    return df

In [3]:
def load_and_preprocess_split(train_path, val_path, test_path):
    """
    Loads three CSVs, cleans them using the predefined pipeline, 
    and returns scaled DataFrames + the fitted scaler.
    """
    
    # 1. Load raw data
    datasets = {
        'train': pd.read_csv(train_path),
        'val': pd.read_csv(val_path),
        'test': pd.read_csv(test_path)
    }
    
    processed_dfs = {}
    
    for name, df in datasets.items():
        # Apply the pipeline we built earlier
        # Note: 'city' and 'to_date' are usually dropped for modeling
        df_clean = (df.pipe(remove_unwanted_columns)  # 1. Strip junk first
                      .pipe(fix_temporal_structure)    # 2. Fix time index
                      .pipe(apply_physical_bounds)    # 3. Clip sensor errors
                      .pipe(handle_missing_values))    # 4. Fill NaNs
        
        if 'time_idx' in df_clean.columns:
            df_clean['time_idx'] = df_clean['time_idx'].astype(int)
        processed_dfs[name] = df_clean

    # 2. Feature Scaling (Standardization)
    # We fit ONLY on train to prevent data leakage
        
    return processed_dfs['train'], processed_dfs['val'], processed_dfs['test']

In [4]:
train_file = "/share/ftrscape/lmiddha/hw/data/Train_data.csv"
val_file = "/share/ftrscape/lmiddha/hw/data/Validation_data.csv"
test_file = "/share/ftrscape/lmiddha/hw/data/Test_data.csv"

In [5]:
train_data, val_data, test_data = load_and_preprocess_split(train_file, val_file, test_file)

In [6]:
feature_cols=[ 'pm25', 'pm10', 'no', 'nh3', 'no2', 'nox', 'so2', 'co', 'ozone', 'bp',
       'wind_speed', 'air_temp', 'humidity', 'rainfall']
scaler = MinMaxScaler()
scaler.fit(train_data[feature_cols])
train_data[feature_cols] = scaler.transform(train_data[feature_cols])
val_data[feature_cols] = scaler.transform(val_data[feature_cols])
test_data[feature_cols] = scaler.transform(test_data[feature_cols])

In [7]:
train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

train_data["city"] = "Delhi"
test_data["city"] = "Delhi"
val_data["city"] = "Delhi"

In [8]:
train_data.head()

,pm25,pm10,no,nh3,no2,nox,so2,co,ozone,bp,wind_speed,air_temp,humidity,rainfall,time_idx,city
0,0.247949,0.320867,0.160433,0.311409,0.232873,0.254292,0.206213,0.160895,0.119595,0.208117,0.016364,0.181511,0.873987,0.0,1,Delhi
1,0.258709,0.344419,0.171318,0.309085,0.228160,0.267750,0.189554,0.182749,0.115936,0.207811,0.017504,0.169319,0.880803,0.0,2,Delhi
2,0.254003,0.330607,0.122315,0.267693,0.188049,0.195340,0.169915,0.128538,0.280472,0.207668,0.013831,0.163223,0.887574,0.0,3,Delhi
3,0.263951,0.302009,0.120730,0.263246,0.157676,0.178996,0.168180,0.124170,0.165346,0.208238,0.015098,0.145396,0.887407,0.0,4,Delhi
4,0.255986,0.271087,0.166050,0.228039,0.135456,0.232222,0.187014,0.147899,0.092966,0.208573,0.009589,0.130063,0.894238,0.0,5,Delhi


In [9]:
test_data.head()

,pm25,pm10,no,nh3,no2,nox,so2,co,ozone,bp,wind_speed,air_temp,humidity,rainfall,time_idx,city
0,0.081741,0.232646,0.266218,0.168774,0.199797,0.292820,0.114118,0.125023,0.031284,0.848249,0.025080,0.347751,0.559211,0.0,54001,Delhi
1,0.090282,0.258060,0.213401,0.183553,0.180402,0.232069,0.108866,0.113532,0.036754,0.846508,0.021630,0.351402,0.534649,0.0,54002,Delhi
2,0.086687,0.222010,0.194948,0.182882,0.160864,0.213559,0.127595,0.103226,0.032776,0.846069,0.021645,0.341923,0.551080,0.0,54003,Delhi
3,0.081731,0.206593,0.196309,0.171079,0.178890,0.210963,0.109311,0.093880,0.038824,0.845452,0.026326,0.344133,0.552164,0.0,54004,Delhi
4,0.079842,0.202226,0.181656,0.168230,0.149773,0.211327,0.097874,0.080843,0.042704,0.845457,0.026797,0.342198,0.544343,0.0,54005,Delhi


In [10]:
val_data.head()

,pm25,pm10,no,nh3,no2,nox,so2,co,ozone,bp,wind_speed,air_temp,humidity,rainfall,time_idx,city
0,0.191747,0.226563,0.148611,0.220066,0.149775,0.161195,0.098529,0.116798,0.032616,0.881160,0.014563,0.175851,0.874877,0.0,52585,Delhi
1,0.185043,0.218535,0.114174,0.218260,0.133553,0.133523,0.091319,0.102693,0.036589,0.880302,0.012512,0.170726,0.878298,0.0,52586,Delhi
2,0.165000,0.193064,0.110916,0.200291,0.118967,0.129111,0.083353,0.088879,0.031809,0.881702,0.012433,0.167241,0.886555,0.0,52587,Delhi
3,0.160067,0.185576,0.124561,0.191399,0.115773,0.137774,0.083461,0.089987,0.030547,0.880580,0.011337,0.160481,0.892008,0.0,52588,Delhi
4,0.149380,0.170272,0.131735,0.187531,0.112380,0.145666,0.068054,0.085447,0.034143,0.880563,0.012740,0.156671,0.895083,0.0,52589,Delhi


In [11]:
import os
os.environ["LIGHTNING_DISABLE_TIPS"] = "1"

In [12]:
import copy
from pathlib import Path
import warnings

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
import numpy as np
import pandas as pd
import torch

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import MultiNormalizer, GroupNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE, PoissonLoss, QuantileLoss
from pytorch_forecasting.data.encoders import MultiNormalizer, TorchNormalizer
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import (
    optimize_hyperparameters,
)
from pytorch_forecasting import Baseline
from pytorch_forecasting.metrics import MAE, RMSE
from lightning.pytorch.callbacks import TQDMProgressBar
import logging
logging.getLogger("lightning").setLevel(logging.ERROR)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
def timeseries_dataset_mt(max_encoder_length, max_prediction_length=24):
    TARGET_COLS = ["pm25", "no2", "co", "ozone"]

    multi_normalizer = MultiNormalizer(
        [TorchNormalizer(method="identity", center=False) for _ in TARGET_COLS]
    )

    training_multi_task = TimeSeriesDataSet(
    train_data,
    time_idx="time_idx",
    target=TARGET_COLS,
    group_ids=["city"],
    min_encoder_length=168,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["city"],
    static_reals=[],
    time_varying_known_categoricals=[],
    variable_groups={},
    time_varying_known_reals=[
        "time_idx", "wind_speed", "air_temp", "humidity", "bp", "rainfall",
    ],
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=[
        "pm10", "pm25", "no", "nh3", "no2", "so2", "co", "ozone", "nox",
    ],
    # identity normalizer because we already scaled with MinMaxScaler
    target_normalizer=multi_normalizer,
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    )

    return training_multi_task

In [14]:
def timeseries_dataset_st(target, max_encoder_length, max_prediction_length=24):

    training_single_task = TimeSeriesDataSet(
        train_data,
        time_idx="time_idx",
        target=target,
        group_ids=["city"],
        min_encoder_length=168,
        max_encoder_length=max_encoder_length,
        min_prediction_length=12,
        max_prediction_length=max_prediction_length,
        static_categoricals=["city"],
        static_reals=[],
        time_varying_known_categoricals=[],
        variable_groups={},
        time_varying_known_reals=[
            "time_idx", "wind_speed", "air_temp", "humidity", "bp", "rainfall",
        ],
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=[
            "pm10", "pm25", "no", "nh3", "no2", "so2", "co", "ozone", "nox",
        ],

        target_normalizer=TorchNormalizer(method="identity", center=False),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    return training_single_task

### Check if data is correct

In [15]:
import numpy as np

expected = np.arange(train_data["time_idx"].iloc[0], train_data["time_idx"].iloc[0] + len(train_data))
(train_data["time_idx"].values == expected).all()

np.True_

In [16]:
last_train_idx = train_data["time_idx"].iloc[-1]
print("Last train time_idx:", last_train_idx)

Last train time_idx: 52584


In [17]:
expected_start = last_train_idx + 1
expected_end = last_train_idx + len(val_data)
expected = np.arange(expected_start, expected_end + 1)  # inclusive
(val_data["time_idx"].values == expected).all()

np.True_

In [18]:
last_val_idx = val_data["time_idx"].iloc[-1]
print("Last val time_idx:", last_val_idx)

Last val time_idx: 54000


In [19]:
expected_start = last_val_idx + 1
expected_end = last_val_idx + len(test_data)
expected = np.arange(expected_start, expected_end + 1)  # inclusive
(test_data["time_idx"].values == expected).all()

np.True_

In [20]:
def evaluate_single_task_results(target, model, max_encoder_length, max_prediction_length=24):
    history = val_data.iloc[-max_encoder_length:].copy()
    step_size   = max_prediction_length
    total_steps = len(test_data)
    targets     = [target]
    
    all_preds   = []
    all_actuals = []
    
    target_idx = feature_cols.index(target)
    
    for start_idx in range(0, total_steps, step_size):
        true_chunk = test_data.iloc[start_idx:start_idx + step_size]
        if len(true_chunk) == 0:
            break
    
        encoder_df = history.iloc[-max_encoder_length:].copy()
        window_df  = pd.concat([encoder_df, true_chunk]).reset_index(drop=True)
    
        global_time_offset        = int(encoder_df["time_idx"].iloc[0])
        window_df["time_idx"]     = range(global_time_offset, global_time_offset + len(window_df))
        window_df["time_idx"]     = window_df["time_idx"].astype(int)
    
        window_dataset = TimeSeriesDataSet.from_dataset(
            timeseries_dataset_st(target, max_encoder_length),
            window_df,
            predict=True,
            stop_randomization=True
        )
        window_dataloader = window_dataset.to_dataloader(
            train=False, batch_size=64, num_workers=1
        )
    
        preds = model.predict(
            window_dataloader,
            mode="prediction",
            trainer_kwargs=dict(accelerator="gpu")
        )
    
        chunk_len  = len(true_chunk)
    
        pred_chunk = preds.reshape(-1).cpu().numpy()[:chunk_len]  
    
        dummy_pred = np.zeros((len(pred_chunk), len(feature_cols)))
        dummy_pred[:, target_idx] = pred_chunk
        pred_original = scaler.inverse_transform(dummy_pred)[:, target_idx]  
    
        actual_chunk = true_chunk[target].values                  
        dummy_act    = np.zeros((chunk_len, len(feature_cols)))
        dummy_act[:, target_idx] = actual_chunk
        act_original = scaler.inverse_transform(dummy_act)[:, target_idx]   
    
        all_preds.append(pred_original)
        all_actuals.append(act_original)
    
        # Update history with true observed values
        history = pd.concat([history, true_chunk])
    
    
    all_preds   = np.concatenate(all_preds)
    all_actuals = np.concatenate(all_actuals)
    
    
    print(f"n_points : {len(all_preds)}")
    print(f"pred  range: {all_preds.min():.2f} – {all_preds.max():.2f}")
    print(f"actual range: {all_actuals.min():.2f} – {all_actuals.max():.2f}")
    
    
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    
    rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
    mae  = mean_absolute_error(all_actuals, all_preds)
    
    print(f"\n── Metrics over full 30-day test set (720 points) ──")
    print(f"{target.upper()}  RMSE: {rmse:.4f} µg/m³  |  MAE: {mae:.4f} µg/m³")



In [21]:
def evaluate_multi_task_results(model, max_encoder_length, max_prediction_length=24):
    history = val_data.iloc[-max_encoder_length:].copy()
    step_size = max_prediction_length
    total_steps = len(test_data)
    targets = ["pm25", "no2", "co", "ozone"]
    
    all_preds   = {t: [] for t in targets}
    all_actuals = {t: [] for t in targets}
    
    for start_idx in range(0, total_steps, step_size):
        true_chunk = test_data.iloc[start_idx:start_idx + step_size]
        if len(true_chunk) == 0:
            break
    
        decoder_df = true_chunk.copy()
    
        encoder_df = history.iloc[-max_encoder_length:].copy()
        window_df  = pd.concat([encoder_df, decoder_df]).reset_index(drop=True)
    
        global_time_offset = int(encoder_df["time_idx"].iloc[0])
        window_df["time_idx"] = range(global_time_offset, global_time_offset + len(window_df))
        window_df["time_idx"] = window_df["time_idx"].astype(int)
    
        window_dataset = TimeSeriesDataSet.from_dataset(
            timeseries_dataset_mt(max_encoder_length, max_prediction_length=24),
            window_df,
            predict=True,
            stop_randomization=True
        )
        window_dataloader = window_dataset.to_dataloader(
            train=False, batch_size=64, num_workers=1
        )
    
        preds = model.predict(
            window_dataloader,
            mode="prediction",
            trainer_kwargs=dict(accelerator="gpu")
        )
    
        chunk_len = len(true_chunk)
    
    
        pred_chunk = np.stack(
            [preds[i].reshape(-1).cpu().numpy()[:chunk_len] for i in range(len(targets))],
            axis=-1
        )  # (chunk_len, 4)
        # Build dummy array for inverse transform
        dummy_pred = np.zeros((pred_chunk.shape[0], len(feature_cols)))
        target_indices = [feature_cols.index(t) for t in targets]
        for i, idx in enumerate(target_indices):
            dummy_pred[:, idx] = pred_chunk[:, i]
        pred_original = scaler.inverse_transform(dummy_pred)[:, target_indices]  # (chunk_len, 4)
    
    
        dummy_act = np.zeros((chunk_len, len(feature_cols)))
        actual_chunk = true_chunk[targets].values        # scaled [0,1] values
        for i, idx in enumerate(target_indices):
            dummy_act[:, idx] = actual_chunk[:, i]
        act_original = scaler.inverse_transform(dummy_act)[:, target_indices]    # (chunk_len, 4)
    
        for i, t in enumerate(targets):
            all_preds[t].append(pred_original[:, i])
            all_actuals[t].append(act_original[:, i])
    
        # Update history with true observed values
        history = pd.concat([history, true_chunk])
    
    
    all_preds   = {t: np.concatenate(all_preds[t])   for t in targets}
    all_actuals = {t: np.concatenate(all_actuals[t]) for t in targets}
    
    
    for t in targets:
        print(f"{t:8s} | n_points: {len(all_preds[t])} "
              f"| pred range: {all_preds[t].min():.2f} – {all_preds[t].max():.2f} "
              f"| true range: {all_actuals[t].min():.2f} – {all_actuals[t].max():.2f}")
    
    
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    
    print("\n── Metrics over full 30-day test set (720 points) ──")
    results = {}
    for t in targets:
        rmse = np.sqrt(mean_squared_error(all_actuals[t], all_preds[t]))
        mae  = mean_absolute_error(all_actuals[t], all_preds[t])
        results[t] = {"RMSE": round(rmse, 4), "MAE": round(mae, 4)}
        print(f"{t:8s}  RMSE: {rmse:.4f}  MAE: {mae:.4f}")
    
    results_df = pd.DataFrame(results).T
    print(results_df)

### Single Task with 7 days lookback

PM2.5

In [22]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/pm25_7days_lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="pm25", model=best_model, max_encoder_length=168)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(i

n_points : 744
pred  range: 30.66 – 154.37
actual range: 14.50 – 195.29

── Metrics over full 30-day test set (720 points) ──
PM25  RMSE: 19.8706 µg/m³  |  MAE: 15.2358 µg/m³


CO

In [23]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/co_7days_lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="co", model=best_model, max_encoder_length=168)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(i

n_points : 744
pred  range: 0.67 – 2.12
actual range: 0.54 – 2.61

── Metrics over full 30-day test set (720 points) ──
CO  RMSE: 0.2298 µg/m³  |  MAE: 0.1593 µg/m³


NO2

In [24]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/no2_7days_lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="no2", model=best_model, max_encoder_length=168)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://p

n_points : 744
pred  range: 18.92 – 68.66
actual range: 14.81 – 71.57

── Metrics over full 30-day test set (720 points) ──
NO2  RMSE: 6.6539 µg/m³  |  MAE: 5.2123 µg/m³


Ozone

In [25]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/ozone_7days_lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="ozone", model=best_model, max_encoder_length=168)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://p

n_points : 744
pred  range: 8.85 – 93.73
actual range: 6.32 – 94.52

── Metrics over full 30-day test set (720 points) ──
OZONE  RMSE: 7.1525 µg/m³  |  MAE: 5.2931 µg/m³


### Single Task with 15 days lookback

CO

In [26]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/co_15days_lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="co", model=best_model, max_encoder_length=360)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://p

n_points : 744
pred  range: 0.68 – 2.26
actual range: 0.54 – 2.61

── Metrics over full 30-day test set (720 points) ──
CO  RMSE: 0.2171 µg/m³  |  MAE: 0.1543 µg/m³


NO2

In [27]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/no2_15days-lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="no2", model=best_model, max_encoder_length=360)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://p

n_points : 744
pred  range: 17.27 – 70.54
actual range: 14.81 – 71.57

── Metrics over full 30-day test set (720 points) ──
NO2  RMSE: 6.3705 µg/m³  |  MAE: 4.6584 µg/m³


Ozone

In [28]:
from pytorch_forecasting.models import TemporalFusionTransformer

best_model = TemporalFusionTransformer.load_from_checkpoint(
    "checkpoints/ozone_15days_lookback.ckpt"
)
best_model.eval()
evaluate_single_task_results(target="ozone", model=best_model, max_encoder_length=360)

/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/usrapps/ftrscape/lmiddha/env_ai/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://p

n_points : 744
pred  range: 9.01 – 86.88
actual range: 6.32 – 94.52

── Metrics over full 30-day test set (720 points) ──
OZONE  RMSE: 6.6686 µg/m³  |  MAE: 5.0523 µg/m³
